## 1. Generate Synthetic Noisy Data (for testing)
Add a helper to simulate measurements with true parameters ζ=0.3, ωₙ=5.0.

In [ ]:
import numpy as np

def generate_synthetic_data(t_min=0.0, t_max=5.0, N=100, noise_std=0.05, zeta_true=0.3, omega_true=5.0):
    t = np.linspace(t_min, t_max, N)
    # Analytical underdamped solution (assume x(0)=1, v(0)=0)
    omega_d = omega_true * np.sqrt(1 - zeta_true**2)
    x = np.exp(-zeta_true * omega_true * t) * (np.cos(omega_d * t) + (zeta_true * omega_true / omega_d) * np.sin(omega_d * t))
    x_noisy = x + noise_std * np.random.randn(N)
    return {'time': t.tolist(), 'displacement': x_noisy.tolist()}

## 2. Adaptive Collocation Points
Replace fixed random collocation with residual-based refinement.

In [2]:
def get_adaptive_collocation(self, n_new=4000):
    # Start with uniform
    t_hat = torch.rand(n_new, 1, device=self.device) * 2 - 1
    t_hat.requires_grad_(True)
    
    # Compute residual magnitude
    r = self.pde_residual(t_hat)
    weights = (r**2).detach().flatten()
    weights = weights / weights.sum()
    
    # Resample proportionally to residual
    indices = torch.multinomial(weights, n_new, replacement=True)
    return t_hat[indices].clone().detach().requires_grad_(True)

## 3. Two-Stage Optimisation (Adam → L-BFGS)
L-BFGS often gives much better final accuracy for PINNs.

In [3]:
from torch.optim import LBFGS

def _run_training(self):
    optimizer_adam = torch.optim.Adam(self.model.parameters(), lr=0.001)
    
    # Phase 1: Adam
    for epoch in range(8000):
        # ... same as before
        optimizer_adam.step()
    
    # Phase 2: L-BFGS
    optimizer_lbfgs = LBFGS(self.model.parameters(), lr=1.0, max_iter=20, tolerance_grad=1e-9)
    
    def closure():
        optimizer_lbfgs.zero_grad()
        loss, _, _, _ = self.total_loss()
        loss.backward()
        return loss
    
    for epoch in range(8000, self.n_epochs):
        optimizer_lbfgs.step(closure)
        # logging...

In [4]:
optimizer_lbfgs


NameError: name 'optimizer_lbfgs' is not defined